# RL 모듈 — 최종 검증 실행 (Colab)

지금까지 찾은 버그 3개(거래비용 2배 계산, 무위험이자율 하드코딩, MACD look-ahead bias)와 VecNormalize가 **전부 한꺼번에 적용된 상태로 Walk-Forward 검증을 다시 돌리는 노트북**입니다. 기존 발표자료의 수치는 이 수정들이 부분적으로만 반영된 상태였어서, 여기 결과가 '진짜 최종' 숫자입니다.

**실행 방법**: 위 메뉴 `런타임 > 모두 실행` (GPU 필요 없음, CPU 런타임으로 충분 — 작은 신경망이라 GPU가 오히려 이득 없음)

**예상 소요시간**: 약 25~35분 (셀 3번)

**주의**: Colab 무료 플랜은 브라우저 탭과 상호작용이 90분 이상 없으면 런타임이 끊길 수 있습니다. 가끔 탭을 확인해주세요.

In [ ]:
# 셀 1: 레포 클론 + 의존성 설치
!git clone https://github.com/kimcanal/robo-advisor-rl.git
%cd robo-advisor-rl
!pip install -q -r rl/requirements.txt

In [ ]:
# 셀 2: 실제 ETF 10종 데이터 받기 (라이선스상 레포엔 CSV가 없어서 직접 받아야 함)
!python -m rl.data.fetch_real_data

In [ ]:
# 셀 3: 최종 Walk-Forward 검증 (2개 윈도우 x 보상함수 3종 x 시드 5개, 전체 버그 수정 반영)
# 약 25~35분 소요
!python -m rl.walk_forward --real --timesteps 120000 --seeds 0,1,2,3,4

In [ ]:
# 셀 4: 결과 확인
import pandas as pd

results = pd.read_csv('rl/outputs/walk_forward/walk_forward_results.csv')
print('=== 윈도우별 요약 ===')
display(results[['window', 'strategy', 'seed', 'cumulative_return', 'sharpe', 'mdd']])

daily = pd.read_csv('rl/outputs/walk_forward/walk_forward_daily.csv')
print(f"\n일별 레코드 수: {len(daily)}")
print(daily['regime'].value_counts())

In [ ]:
# 셀 5 (선택): 결과 CSV를 로컬로 다운로드
from google.colab import files
files.download('rl/outputs/walk_forward/walk_forward_results.csv')
files.download('rl/outputs/walk_forward/walk_forward_daily.csv')